# SEHS4696 Group Project — Job Salary Prediction

**Real-world problem:** A recruitment agency needs to provide accurate salary ranges across different industries, serving both employers (budgeting & compensation planning) and job seekers (setting realistic salary expectations during negotiations).

**Task type:** Supervised regression — predict salary as a continuous value and present results as a range (point estimate ± MAE) for advisory use.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
dataset = pd.read_csv('/content/drive/MyDrive/job_salary_prediction_dataset.csv')

print('Features:', dataset.columns.tolist())
print('Size (rows, columns):', dataset.shape)
print('\nData Types:')
print(dataset.dtypes)
dataset.head()

## 1. Exploratory Data Analysis (EDA)

In [ ]:
print('Summary Statistics:')
display(dataset.describe())
print('\nMissing Values per Column:')
print(dataset.isnull().sum())
print('\nSample Rows:')
display(dataset.head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.histplot(dataset['salary'], bins=50, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Salary Distribution', fontsize=14)
axes[0].set_xlabel('Salary ($)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Numerical columns derived from data
numerical_cols = dataset.select_dtypes(include='number').columns.tolist()
corr_df = dataset[numerical_cols].copy()

# Ordinal encode education_level — has a clear natural order (more education → higher salary expectation)
# Nominal columns (job_title, industry, location) are excluded: label encoding would imply false ordering
edu_order = {'High School': 1, 'Bachelor': 2, 'Master': 3, 'PhD': 4}
corr_df['education_level'] = dataset['education_level'].map(edu_order)

corr = corr_df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1])
axes[1].set_title(f'Correlation Heatmap ({len(corr_df.columns)} features)', fontsize=14)

plt.tight_layout()
plt.show()

print('Features in heatmap:', corr_df.columns.tolist())

In [ ]:
plt.figure(figsize=(14, 6))
industry_avg = dataset.groupby('industry')['salary'].mean().sort_values(ascending=False)
sns.barplot(x=industry_avg.index, y=industry_avg.values, hue=industry_avg.index, palette='Blues_d', legend=False)
plt.title('Average Salary by Industry', fontsize=16)
plt.xlabel('Industry', fontsize=12)
plt.ylabel('Average Salary ($)', fontsize=12)
plt.xticks(rotation=30, ha='right')
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

edu_avg = dataset.groupby('education_level')['salary'].mean().sort_values(ascending=False)
sns.barplot(x=edu_avg.index, y=edu_avg.values, hue=edu_avg.index, palette='Greens_d', legend=False, ax=axes[0])
axes[0].set_title('Average Salary by Education Level', fontsize=14)
axes[0].set_xlabel('Education Level', fontsize=12)
axes[0].set_ylabel('Average Salary ($)', fontsize=12)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

remote_avg = dataset.groupby('remote_work')['salary'].mean().sort_values(ascending=False)
sns.barplot(x=remote_avg.index, y=remote_avg.values, hue=remote_avg.index, palette='Oranges_d', legend=False, ax=axes[1])
axes[1].set_title('Average Salary by Work Arrangement', fontsize=14)
axes[1].set_xlabel('Work Arrangement', fontsize=12)
axes[1].set_ylabel('Average Salary ($)', fontsize=12)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

## 2. Data Preprocessing

In [ ]:
# One-hot encode all categorical columns
# remote_work is kept — work arrangement (remote/hybrid/on-site) directly affects salary
categorical_cols = dataset.select_dtypes(include=['object']).columns
dataset_encoded = pd.get_dummies(dataset, columns=categorical_cols, drop_first=True, dtype=int)

print('Original columns:', dataset.shape[1])
print('Encoded columns: ', dataset_encoded.shape[1])
print('Encoded shape:   ', dataset_encoded.shape)
display(dataset_encoded.head(3))

In [ ]:
print('Null check after encoding:')
print(dataset_encoded.isnull().sum().sum(), 'missing values')

In [ ]:
y = dataset_encoded['salary']
X = dataset_encoded.drop('salary', axis=1)

# 60/40 split — large dataset (250k rows) means 100k test samples is a robust evaluation set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

print(f'Training set: {X_train.shape[0]:,} samples')
print(f'Test set:     {X_test.shape[0]:,} samples')
print(f'Features:     {X_train.shape[1]}')

## 3. Model Training

Four models are trained and compared:
- **Ridge Regression** — regularized linear baseline; L2 penalty stabilises coefficients when there are many OHE features
- **Random Forest** — bagging ensemble; handles non-linearity; robust to outliers; no scaling required
- **Gradient Boosting** — sequential boosting; captures complex feature interactions
- **XGBoost** — optimized gradient boosting; typically highest accuracy on tabular data with faster training

In [ ]:
print('Training Ridge Regression (alpha=1.0)...')
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)
ridge_mae = mean_absolute_error(y_test, ridge_pred)
ridge_r2 = r2_score(y_test, ridge_pred)
print(f'R²: {ridge_r2:.4f}  |  MAE: ${ridge_mae:,.2f}')

plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=ridge_pred, alpha=0.4)
p1, p2 = max(max(ridge_pred), max(y_test)), min(min(ridge_pred), min(y_test))
plt.plot([p1, p2], [p1, p2], 'r--', linewidth=2, label='Perfect Prediction')
plt.title('Ridge Regression: Actual vs. Predicted Salary', fontsize=14)
plt.xlabel('Actual Salary ($)')
plt.ylabel('Predicted Salary ($)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': ridge_model.coef_})
coef_df['Abs'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values('Abs', ascending=False).drop('Abs', axis=1)
print('--- Top 10 Most Impactful Features (Ridge Regression) ---')
print(coef_df.head(10).to_string(index=False))

In [ ]:
print('Training Random Forest (100 trees, all CPU cores)...')
random_forest_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
random_forest_model.fit(X_train, y_train)
rf_pred = random_forest_model.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)
print(f'R²: {rf_r2:.4f}  |  MAE: ${rf_mae:,.2f}')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.scatterplot(x=y_test, y=rf_pred, alpha=0.4, ax=axes[0])
p1, p2 = max(max(rf_pred), max(y_test)), min(min(rf_pred), min(y_test))
axes[0].plot([p1, p2], [p1, p2], 'r--', linewidth=2)
axes[0].set_title('Random Forest: Actual vs. Predicted', fontsize=14)
axes[0].set_xlabel('Actual Salary ($)')
axes[0].set_ylabel('Predicted Salary ($)')
axes[0].grid(True)

residuals = y_test - rf_pred
sns.scatterplot(x=rf_pred, y=residuals, alpha=0.4, ax=axes[1])
axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1].set_title('Random Forest: Residual Plot', fontsize=14)
axes[1].set_xlabel('Predicted Salary ($)')
axes[1].set_ylabel('Residuals ($)')
axes[1].grid(True)

plt.suptitle('Random Forest Performance', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
print('Training Gradient Boosting (100 estimators, lr=0.1)...')
gradient_boosting_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gradient_boosting_model.fit(X_train, y_train)
gb_pred = gradient_boosting_model.predict(X_test)
gb_mae = mean_absolute_error(y_test, gb_pred)
gb_r2 = r2_score(y_test, gb_pred)
print(f'R²: {gb_r2:.4f}  |  MAE: ${gb_mae:,.2f}')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.scatterplot(x=y_test, y=gb_pred, alpha=0.4, ax=axes[0])
p1, p2 = max(max(gb_pred), max(y_test)), min(min(gb_pred), min(y_test))
axes[0].plot([p1, p2], [p1, p2], 'r--', linewidth=2)
axes[0].set_title('Gradient Boosting: Actual vs. Predicted', fontsize=14)
axes[0].set_xlabel('Actual Salary ($)')
axes[0].set_ylabel('Predicted Salary ($)')
axes[0].grid(True)

residuals = y_test - gb_pred
sns.scatterplot(x=gb_pred, y=residuals, alpha=0.4, ax=axes[1])
axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1].set_title('Gradient Boosting: Residual Plot', fontsize=14)
axes[1].set_xlabel('Predicted Salary ($)')
axes[1].set_ylabel('Residuals ($)')
axes[1].grid(True)

plt.suptitle('Gradient Boosting Performance', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
from xgboost import XGBRegressor

print('Training XGBoost (100 estimators, lr=0.1)...')
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_r2 = r2_score(y_test, xgb_pred)
print(f'R²: {xgb_r2:.4f}  |  MAE: ${xgb_mae:,.2f}')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.scatterplot(x=y_test, y=xgb_pred, alpha=0.4, ax=axes[0])
p1, p2 = max(max(xgb_pred), max(y_test)), min(min(xgb_pred), min(y_test))
axes[0].plot([p1, p2], [p1, p2], 'r--', linewidth=2)
axes[0].set_title('XGBoost: Actual vs. Predicted', fontsize=14)
axes[0].set_xlabel('Actual Salary ($)')
axes[0].set_ylabel('Predicted Salary ($)')
axes[0].grid(True)

residuals = y_test - xgb_pred
sns.scatterplot(x=xgb_pred, y=residuals, alpha=0.4, ax=axes[1])
axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1].set_title('XGBoost: Residual Plot', fontsize=14)
axes[1].set_xlabel('Predicted Salary ($)')
axes[1].set_ylabel('Residuals ($)')
axes[1].grid(True)

plt.suptitle('XGBoost Performance', fontsize=16)
plt.tight_layout()
plt.show()

## 4. Model Evaluation & Comparison

In [ ]:
results_data = {
    'Model': ['Ridge Regression', 'Random Forest', 'Gradient Boosting', 'XGBoost'],
    'R-squared (R²)': [ridge_r2, rf_r2, gb_r2, xgb_r2],
    'MAE ($)': [ridge_mae, rf_mae, gb_mae, xgb_mae]
}
results_df = pd.DataFrame(results_data)
results_df = results_df.sort_values(
    by=['R-squared (R²)', 'MAE ($)'], ascending=[False, True]
).reset_index(drop=True)

print('--- Model Comparison Scorecard ---')
display(results_df)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = ['#4472C4', '#ED7D31', '#70AD47', '#FFC000']
bars = axes[0].bar(results_df['Model'], results_df['R-squared (R²)'], color=colors)
axes[0].set_title('R² Score — higher is better', fontsize=13)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('R²')
axes[0].tick_params(axis='x', rotation=15)
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)

bars2 = axes[1].bar(results_df['Model'], results_df['MAE ($)'], color=colors)
axes[1].set_title('MAE — lower is better', fontsize=13)
axes[1].set_ylabel('MAE ($)')
axes[1].tick_params(axis='x', rotation=15)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200,
                 f'${bar.get_height():,.0f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Model Comparison', fontsize=16)
plt.tight_layout()
plt.show()

## 5. Feature Importance

In [ ]:
# Dynamically pick the best model from the comparison results (highest R²)
model_map = {
    'Ridge Regression':    ridge_model,
    'Random Forest':       random_forest_model,
    'Gradient Boosting':   gradient_boosting_model,
    'XGBoost':             xgb_model
}
best_model_name = results_df.iloc[0]['Model']
best_model = model_map[best_model_name]
best_mae = results_df.iloc[0]['MAE ($)']
print(f'Best model: {best_model_name}  (R²: {results_df.iloc[0]["R-squared (R²)"]:.4f}, MAE: ${best_mae:,.2f})')

# Ridge uses |coefficients| as importance; tree models use feature_importances_
if hasattr(best_model, 'feature_importances_'):
    importance_values = best_model.feature_importances_
else:
    importance_values = np.abs(best_model.coef_)

# OHE splits each categorical column into many binary columns (e.g. job_title_Data Analyst,
# job_title_Business Analyst ...). Aggregate importances back to the original column level
# so the result reflects dataset columns, not OHE dummies.
original_cat_cols = dataset.select_dtypes(include='object').columns.tolist()
original_num_cols = [c for c in dataset.select_dtypes(include='number').columns if c != 'salary']

def original_col(feature):
    for col in original_cat_cols:
        if feature.startswith(col + '_'):
            return col
    return feature  # numerical columns keep their name

aggregated = {}
for feat, imp in zip(X.columns, importance_values):
    orig = original_col(feat)
    aggregated[orig] = aggregated.get(orig, 0) + imp

feature_importance_df = pd.DataFrame(
    list(aggregated.items()), columns=['Feature', 'Importance']
).sort_values('Importance', ascending=False).reset_index(drop=True)

print('\n--- Feature Importance by Original Dataset Column ---')
print(feature_importance_df.to_string(index=False))

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', hue='Feature',
            data=feature_importance_df, palette='viridis', legend=False)
plt.title(f'Feature Importance by Original Column ({best_model_name})', fontsize=14)
plt.xlabel('Aggregated Importance Score')
plt.ylabel('Dataset Column')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 6. Salary Range Advisory — Live Demo

Edit the `candidate_profile` dict below and run the cell to get a salary range estimate for any job profile.

The output is presented as **point estimate ± MAE** — the format a recruitment agency would quote to employers and job seekers.

In [ ]:
candidate_profile = {
    'job_title':       'Data Analyst',
    'experience_years': 5,
    'education_level': 'Master',
    'skills_count':    10,
    'industry':        'Finance',
    'company_size':    'Large',
    'location':        'USA',
    'remote_work':     'Hybrid',
    'certifications':  1
}

raw_sample = pd.DataFrame([candidate_profile])

# Encode using the same OHE strategy; reindex aligns to training columns (missing cols → 0)
cat_cols = raw_sample.select_dtypes(include=['object']).columns
sample_encoded = pd.get_dummies(raw_sample, columns=cat_cols, dtype=int)
sample_encoded = sample_encoded.reindex(columns=X.columns, fill_value=0)

predicted_salary = best_model.predict(sample_encoded)[0]
salary_low  = predicted_salary - best_mae
salary_high = predicted_salary + best_mae

role   = candidate_profile['job_title']
ind    = candidate_profile['industry']
exp    = candidate_profile['experience_years']
edu    = candidate_profile['education_level']
work   = candidate_profile['remote_work']
loc    = candidate_profile['location']
skills = candidate_profile['skills_count']
certs  = candidate_profile['certifications']

print('=' * 54)
print('   RECRUITMENT AGENCY — SALARY ADVISORY REPORT')
print('=' * 54)
print(f'  Role:          {role}')
print(f'  Industry:      {ind}')
print(f'  Experience:    {exp} years')
print(f'  Education:     {edu}')
print(f'  Work type:     {work}')
print(f'  Location:      {loc}')
print(f'  Skills: {skills}  |  Certifications: {certs}')
print('-' * 54)
print(f'  Model used:      {best_model_name}')
print(f'  Predicted Salary Range:')
print(f'    ${salary_low:>10,.0f}  to  ${salary_high:,.0f}')
print(f'  Point estimate:  ${predicted_salary:,.0f}')
print(f'  Margin:          ± ${best_mae:,.0f}  (model MAE)')
print('=' * 54)